In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import MeanShift
import random

In [2]:
np.random.seed(84)
random.seed(84)

In [3]:
# pd.set_option('display.float_format', '{:,.4E}'.format)

In [4]:
pd.options.display.max_seq_items = 2000

In [5]:
reconciled_data = pd.read_pickle('reconciled_data.pkl')

kinetic_data = reconciled_data.loc[reconciled_data.CINÉTICA == True]

In [6]:
kinetic_data['P (bar)'] = kinetic_data.P_abs_Pa / 1e5

In [7]:
X = kinetic_data[['T_R_C', 'P (bar)', 'F_H2_e_mol_s', 'F_CO_e_mol_s']]
clustering = MeanShift(bandwidth=2).fit(X)

kinetic_data['cluster'] = clustering.labels_

In [8]:
exp_design = kinetic_data[['t_h', 'T_R_C', 'P (bar)', 'F_H2_e_mol_s', 'F_CO_e_mol_s', 'cluster']]

In [9]:
exp_design.groupby('cluster').min()

,t_h,T_R_C,P (bar),F_H2_e_mol_s,F_CO_e_mol_s
cluster,,,,,
0,125.083333,262.0,20.609977,0.000053,0.000026
1,107.833333,253.0,20.885768,0.000057,0.000024
2,138.750000,271.0,21.437348,0.000050,0.000026
3,35.250000,244.0,16.611018,0.000077,0.000043
4,19.583333,243.0,21.851034,0.000077,0.000046
5,46.750000,222.0,16.886809,0.000098,0.000067
6,24.750000,211.0,21.919981,0.000104,0.000043
7,5.500000,233.0,22.333667,0.000091,0.000030
8,13.000000,222.0,21.919981,0.000084,0.000048


In [10]:
exp_design_summary = exp_design.groupby('cluster').mean()
exp_design_summary['t_h'] = exp_design.groupby('cluster').min().t_h
exp_design_summary = exp_design_summary.sort_values('t_h')
exp_design_summary.columns = [r'$t$ (h)', r'$T$ ($\mathrm{^o}$C)', r'$P$ (bar)', 
                                   r'$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$', r'$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$']
exp_design_summary

,$t$ (h),$T$ ($\mathrm{^o}$C),$P$ (bar),"$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$","$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$"
cluster,,,,,
7,5.500000,233.000000,22.983744,0.000095,0.000037
8,13.000000,222.000000,22.629156,0.000095,0.000055
4,19.583333,243.375000,22.618075,0.000083,0.000047
6,24.750000,211.000000,22.618075,0.000112,0.000053
3,35.250000,244.000000,17.378060,0.000092,0.000048
15,39.750000,211.000000,16.714440,0.000153,0.000057
9,43.250000,233.000000,17.645232,0.000099,0.000043
5,46.750000,222.000000,17.791746,0.000108,0.000068
14,55.333333,211.000000,26.022362,0.000133,0.000067


In [11]:
# % deviation on the means
100*exp_design.groupby('cluster').std() / exp_design.groupby('cluster').mean()

,t_h,T_R_C,P (bar),F_H2_e_mol_s,F_CO_e_mol_s
cluster,,,,,
0,2.399311,0.198911,1.115025,7.881686,24.085245
1,3.980872,0.207908,1.180799,2.170620,26.816312
2,0.997571,0.000000,0.160468,14.641029,14.641029
3,44.975870,0.000000,5.548180,12.319152,7.561185
4,78.558448,0.212655,4.269036,8.925581,1.086366
5,30.855159,0.000000,5.438935,8.595659,2.188256
6,71.878761,0.000000,3.492549,8.589585,23.640474
7,107.582882,0.000000,3.253397,3.417056,25.036316
8,99.234001,0.000000,3.573672,18.298261,18.865114


In [12]:
exp_design_summary.columns

Index(['$t$ (h)', '$T$ ($\mathrm{^o}$C)', '$P$ (bar)',
       '$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$',
       '$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$'],
      dtype='str')

In [13]:
deactivation_data = reconciled_data.loc[reconciled_data.DESATIVAÇÃO == True]
deactivation_data['P (bar)'] = deactivation_data.P_abs_Pa / 1e5

In [14]:
X = deactivation_data[['t_h']]
clustering = MeanShift(bandwidth=17).fit(X)

deactivation_data['cluster_deact_time'] = clustering.labels_

In [15]:
deactivation_data['cluster_deact_time'].unique()

array([0, 3, 5, 2, 4, 1])

In [16]:
exp_design_deac = deactivation_data[['t_h', 'T_R_C', 'P (bar)', 'F_H2_e_mol_s', 'F_CO_e_mol_s', 'cluster_deact_time']]
exp_design_summary_deac = exp_design_deac.groupby('cluster_deact_time').mean()
exp_design_summary_deac['t_h'] = exp_design_deac.groupby('cluster_deact_time').min().t_h
exp_design_summary_deac = exp_design_summary_deac.sort_values('t_h')
exp_design_summary_deac.columns = [r'$t$ (h)', r'$T$ ($\mathrm{^o}$C)', r'$P$ (bar)', 
                                   r'$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$', r'$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$']
exp_design_summary_deac

,$t$ (h),$T$ ($\mathrm{^o}$C),$P$ (bar),"$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$","$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$"
cluster_deact_time,,,,,
0,2.250000,233.0,22.425597,0.000091,0.000029
3,28.416667,233.0,21.868271,0.000094,0.000030
5,51.250000,233.0,21.988929,0.000102,0.000033
2,70.083333,233.0,22.195772,0.000096,0.000031
4,102.750000,233.0,21.000680,0.000104,0.000034
1,147.916667,232.0,21.178795,0.000092,0.000035


In [17]:
exp_design_summary['Deactivation'] = False
exp_design_summary['Kinetics'] = True
exp_design_summary_deac['Deactivation'] = True
exp_design_summary_deac['Kinetics'] = False

In [18]:
exp_design_overall = pd.concat((exp_design_summary, exp_design_summary_deac))

In [19]:
exp_design_overall = exp_design_overall.sort_values(r'$t$ (h)')

In [20]:
exp_design_overall[r'$\mathrm{H_2/CO}$'] = (
    exp_design_overall[r'$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$'] / 
    exp_design_overall[r'$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$']
)

In [21]:
exp_design_overall[r'$W$ (g)'] = 5.12

In [22]:
exp_design_overall = exp_design_overall[[r'$t$ (h)', r'$W$ (g)', r'$T$ ($\mathrm{^o}$C)', r'$P$ (bar)',
       r'$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$',
       r'$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$', r'$\mathrm{H_2/CO}$', 'Kinetics', 'Deactivation']]

In [23]:
exp_design_overall

,$t$ (h),$W$ (g),$T$ ($\mathrm{^o}$C),$P$ (bar),"$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$","$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$",$\mathrm{H_2/CO}$,Kinetics,Deactivation
0,2.250000,5.12,233.000000,22.425597,0.000091,0.000029,3.087668,False,True
7,5.500000,5.12,233.000000,22.983744,0.000095,0.000037,2.535104,True,False
8,13.000000,5.12,222.000000,22.629156,0.000095,0.000055,1.740614,True,False
4,19.583333,5.12,243.375000,22.618075,0.000083,0.000047,1.766460,True,False
6,24.750000,5.12,211.000000,22.618075,0.000112,0.000053,2.125039,True,False
3,28.416667,5.12,233.000000,21.868271,0.000094,0.000030,3.107949,False,True
3,35.250000,5.12,244.000000,17.378060,0.000092,0.000048,1.914665,True,False
15,39.750000,5.12,211.000000,16.714440,0.000153,0.000057,2.692866,True,False
9,43.250000,5.12,233.000000,17.645232,0.000099,0.000043,2.270612,True,False
5,46.750000,5.12,222.000000,17.791746,0.000108,0.000068,1.593390,True,False


In [30]:
exp_design_overall[r'$T$ ($\mathrm{^o}$C)'] = exp_design_overall[r'$T$ ($\mathrm{^o}$C)'] - exp_design_overall[r'$T$ ($\mathrm{^o}$C)'] % 10

In [31]:
print(exp_design_overall.to_latex(index=False, 
                                  formatters = {
                                                r'$t$ (h)'                : lambda x: f"\\num{{{x:.1f}}}",
                                                r'$W$ (g)'                : lambda x: f"\\num{{{x:.2f}}}",
                                                r'$T$ ($\mathrm{^o}$C)'   : lambda x: f"\\num{{{x:.0f}}}",
                                                r'$P$ (bar)'              : lambda x: f"\\num{{{x:.0f}}}",
                                                r'$F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$'   : lambda x: f"\\num{{{x:.2E}}}",
                                                r'$F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$'    : lambda x: f"\\num{{{x:.2E}}}",
                                                r'$\mathrm{H_2/CO}$'      : lambda x: f"\\num{{{x:.1f}}}",
                                            }
                                 ))

\begin{tabular}{rrrrrrrrr}
\toprule
$t$ (h) & $W$ (g) & $T$ ($\mathrm{^o}$C) & $P$ (bar) & $F_\mathrm{H_2, in}~\mathrm{(mol~s^{-1})}$ & $F_\mathrm{CO, in}~\mathrm{(mol~s^{-1})}$ & $\mathrm{H_2/CO}$ & Kinetics & Deactivation \\
\midrule
\num{2.2} & \num{5.12} & \num{230} & \num{22} & \num{9.07E-05} & \num{2.94E-05} & \num{3.1} & False & True \\
\num{5.5} & \num{5.12} & \num{230} & \num{23} & \num{9.47E-05} & \num{3.74E-05} & \num{2.5} & True & False \\
\num{13.0} & \num{5.12} & \num{220} & \num{23} & \num{9.51E-05} & \num{5.46E-05} & \num{1.7} & True & False \\
\num{19.6} & \num{5.12} & \num{240} & \num{23} & \num{8.32E-05} & \num{4.71E-05} & \num{1.8} & True & False \\
\num{24.8} & \num{5.12} & \num{210} & \num{23} & \num{1.12E-04} & \num{5.29E-05} & \num{2.1} & True & False \\
\num{28.4} & \num{5.12} & \num{230} & \num{22} & \num{9.42E-05} & \num{3.03E-05} & \num{3.1} & False & True \\
\num{35.2} & \num{5.12} & \num{240} & \num{17} & \num{9.15E-05} & \num{4.78E-05} & \num{1.9} & True 